In [4]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import torch
import torch.nn as nn
from torch.optim import Adam
from src.models.vit_transformer import get_vit
from src.datasets.doggie_loader import get_doggie_dataset, create_doggie_dataloaders, get_default_transforms
from tqdm import tqdm

In [6]:
# Get the datasets (already split from before)
train_dataset, val_dataset, test_dataset, class_names, num_classes = get_doggie_dataset(
    data_path="/Users/cindychen/Desktop/EE562/EE562 Assignments/EE562-Classifiers/data/dataset_split"
)

# dataloaders
train_loader, val_loader, test_loader = create_doggie_dataloaders(
    train_dataset, 
    val_dataset, 
    test_dataset, 
    batch_size=32, 
    num_workers=0
)

print(f"Number of classes: {num_classes}")

Number of classes: 120
Train samples: 16418
Val samples: 2009
Test samples: 2153

Using batch size: 32
Train batches: 514
Validation batches: 63
Test batches: 68
Number of classes: 120


In [7]:
num_classes = 27
model = get_vit(num_classes, pretrained=False)

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device)

ByobNet(
  (stem): ConvNormAct(
    (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
  )
  (stages): Sequential(
    (0): Sequential(
      (0): BottleneckBlock(
        (conv1_1x1): ConvNormAct(
          (conv): Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
        )
        (conv2_kxk): ConvNormAct(
          (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
   

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler('mps') if torch.backends.mps.is_available() else None

In [10]:
# Initialize tracking variable
best_loss = float('inf')  
checkpoint_dir = '/Users/cindychen/Desktop/EE562/EE562 Assignments/EE562-Classifiers/outputs/doggie/checkpoints/'
os.makedirs(checkpoint_dir, exist_ok=True)

# load preexisting checkpoint if available
latest_path = os.path.join(checkpoint_dir, 'vit_doggie_latest.pth')
if os.path.exists(latest_path):
    checkpoint = torch.load(latest_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    best_loss = checkpoint.get('loss', float('inf'))
    print(f"Loaded checkpoint from {latest_path}, epoch {checkpoint.get('epoch', '?')}, loss {best_loss:.4f}")

for epoch in range(10):
    model.train()
    loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/10', unit='batch')
    running_loss = 0.0
    
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast(device_type='mps', dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = running_loss / len(train_loader)
    print(f'\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}')


    checkpoint_data = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'loss': avg_loss,
    }

    # save latest checkpoint every epoch
    torch.save(checkpoint_data, latest_path)

    # 2. save best epoch if improved
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_path = os.path.join(checkpoint_dir, 'vit_doggie_best.pth')
        torch.save(checkpoint_data, best_path)
        print(f"⭐ New best model saved with loss: {best_loss:.4f}")

Epoch 1/10: 100%|██████████| 514/514 [07:44<00:00,  1.11batch/s, loss=1.6719]



Epoch 1 Average Loss: 0.7919
⭐ New best model saved with loss: 0.7919


Epoch 2/10: 100%|██████████| 514/514 [06:48<00:00,  1.26batch/s, loss=0.0000]



Epoch 2 Average Loss: 0.7786
⭐ New best model saved with loss: 0.7786


Epoch 3/10: 100%|██████████| 514/514 [06:45<00:00,  1.27batch/s, loss=1.4785]


...


Epoch 8 Average Loss: 0.6631
⭐ New best model saved with loss: 0.6631


Epoch 9/10: 100%|██████████| 514/514 [08:18<00:00,  1.03batch/s, loss=0.0000]



Epoch 9 Average Loss: 0.6412
⭐ New best model saved with loss: 0.6412


Epoch 10/10: 100%|██████████| 514/514 [08:22<00:00,  1.02batch/s, loss=0.0000]



Epoch 10 Average Loss: 0.6290
⭐ New best model saved with loss: 0.6290
